> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 12 · Notebook 02 — The strategy creator and screeners

**Sessions:** S3–S4 (Strategy creator from library functions · Instrument selection & screeners) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Turn a YAML strategy config into a signal function from registered indicators and actions.
2. Reject a bad config before it runs, with every error listed.
3. Screen a point-in-time universe with composable filters.
4. Measure universe turnover, and store screen results so a live universe can be reproduced.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. Strategies as data

A strategy is a YAML config, not code: conditions are trees of `all`, `any`, `not` and leaf actions (`crossover`, `gt`, `lt`) over series (registered indicators, columns or constants). Build it recursively: `all` → element-wise AND of its parts (`np.logical_and.reduce`), `any` → OR (`np.logical_or.reduce`), `not` → negation. Each node returns a function of the data.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def build_condition(spec, registry=p.REGISTRY):
    if "all" in spec:
        parts = [build_condition(s, registry) for s in spec["all"]]
        return lambda d: np.logical_and.reduce([f(d) for f in parts])
    if "any" in spec:
        parts = [build_condition(s, registry) for s in spec["any"]]
        return lambda d: np.logical_or.reduce([f(d) for f in parts])
    if "not" in spec:
        inner = build_condition(spec["not"], registry)
        return lambda d: ~inner(d)
    fn = registry["actions"][spec["fn"]]
    args = [p.build_series(a, registry) for a in spec["args"]]
    return lambda d: fn(*[a(d) for a in args])

CONFIG = """
name: ema_cross_rsi
entry:
  all:
    - fn: crossover
      args: [{ind: ema, params: {n: 20}}, {ind: ema, params: {n: 50}}]
    - not: {fn: gt, args: [{ind: rsi, params: {n: 14}}, 70]}
exit: {fn: lt, args: [close, {ind: sma, params: {n: 20}}]}
sizing: {risk_per_trade: 0.005}
"""
import yaml

cfg = yaml.safe_load(CONFIG)
bars = p.ohlcv()
specs = [cfg["entry"], cfg["exit"], {"any": [cfg["exit"], {"fn": "gt", "args": ["close", 150]}]}]
mine = [p.attempt(lambda s=s: build_condition(s)(bars)) for s in specs]
mine = p.check("build_condition", mine, [p.build_condition(s)(bars) for s in specs])
print(f"entry signals: {int(mine[0].sum())} of {len(bars)} days")

## 2. Validate before running

A config is checked against the registry before any backtest or live run: unknown actions and indicators, unknown parameters, and parameters outside their allowed range (`p.REGISTRY["params"]`, e.g. `ema.n` in [2, 500]). The part that checks one series: for each parameter, report `"unknown parameter <ind>.<name>"` if it has no range, else `"<ind>.<name>=<v> outside [lo, hi]"` if it is out of range.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def walk_series(s, errors, registry=p.REGISTRY):
    if not isinstance(s, dict):
        return                                                       # a column name or a constant
    ind = s.get("ind")
    if ind not in registry["indicators"]:
        errors.append(f"unknown indicator {ind}")
        return
    ranges = registry["params"].get(ind, {})
    for name, v in s.get("params", {}).items():
        if name not in ranges:
            errors.append(f"unknown parameter {ind}.{name}")
        elif not ranges[name][0] <= v <= ranges[name][1]:
            errors.append(f"{ind}.{name}={v} outside [{ranges[name][0]}, {ranges[name][1]}]")

series = [{"ind": "ema", "params": {"n": 20}}, {"ind": "ema", "params": {"n": 1000}}, {"ind": "ema", "params": {"window": 5}},
          {"ind": "macd"}, "close", 70, {"ind": "rsi", "params": {"n": 1}}]
mine = []
for s in series:
    errs = []
    mine.append(p.attempt(lambda: walk_series(s, errs) or errs))
bad = yaml.safe_load(CONFIG)
bad["entry"]["all"][0]["args"][1]["params"]["n"] = 1000
bad["entry"]["all"][1] = {"fn": "less", "args": [1, 2]}
del bad["sizing"]
reference = []
for s in series:
    cfg_one = {"name": "x", "exit": {}, "sizing": {}, "entry": {"fn": "gt", "args": [s, 0]}}
    reference.append(p.validate_config(cfg_one))
mine = p.check("walk_series", mine, reference)
print("errors in a broken config:", p.validate_config(bad))
try:
    p.load_strategy(yaml.safe_dump(bad))
except ValueError as err:
    print("load_strategy refuses it:", err)

A valid config also gets a **hash** (`p.config_hash`: the first 12 hex digits of the sha256 of the sorted JSON), stored with every trade, so any fill can be traced to the exact parameters that produced it:

In [ ]:
cfg, entry, h = p.load_strategy(CONFIG)
signal = entry(bars)
print(f"config hash {h}; {int(signal.sum())} entries")
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(bars.index, bars["close"], lw=0.8)
ax.scatter(bars.index[signal], bars["close"][signal], color="#1baf7a", s=20, zorder=3, label="entry")
ax.legend(); ax.set_title(f"{cfg['name']} (config {h})"); plt.show()

## 3. Screeners

A screen is a list of filter factories (`p.min_adv`, `p.price_between`, `p.max_spread`, `p.sector_in`, `p.min_option_oi`, `p.shortable`), each mapping a snapshot to a boolean Series. Keep the symbols that pass **all** filters (`np.logical_and.reduce`, all True when there are none), sort by `adv_usd` descending then `symbol`, and return at most `limit`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def screen(df, filters, limit=None):
    mask = np.logical_and.reduce([f(df) for f in filters]) if filters else np.ones(len(df), bool)
    out = df[mask].sort_values(["adv_usd", "symbol"], ascending=[False, True])["symbol"].tolist()
    return out if limit is None else out[:limit]

snap = p.universe_snapshot()                                         # one day's point-in-time universe (300 names)
filters = [p.min_adv(2e7), p.price_between(10, 500), p.max_spread(10), p.shortable()]
cases = [(snap, filters), (snap, filters, 20), (snap, [p.sector_in(["tech"]), p.min_option_oi(5000)], 10), (snap, [])]
mine = [p.attempt(screen, *c) for c in cases]
mine = p.check("screen", mine, [p.screen(*c) for c in cases])
print(f"{len(snap)} names → {len(mine[0])} pass the liquid, shortable screen;  top 5 by dollar volume: {mine[1][:5]}")

## 4. Turnover and reproducibility

Every name that enters the universe is a trade. **Universe turnover** is the share of today's universe that wasn't in yesterday's. How much does day-to-day noise in liquidity and spreads move it?

In [ ]:
rows = {}
for noise in (0.05, 0.15, 0.30):
    tomorrow = p.next_snapshot(snap, noise=noise)
    rows[f"noise {noise:.0%}"] = {"names today": len(p.screen(snap, filters)), "names tomorrow": len(p.screen(tomorrow, filters)),
                                   "turnover": p.universe_turnover(p.screen(snap, filters), p.screen(tomorrow, filters))}
display(pd.DataFrame(rows).T.round(3))
store = p.UniverseStore()
store.save("2025-06-02", "liquid_shortable", {"filters": "adv>=2e7, 10<=px<=500, spread<=10, shortable"}, p.screen(snap, filters))
print("as of 2025-06-05 the live universe came from:", {k: (v if k != "symbols" else f"{len(v)} symbols") for k, v in store.get("2025-06-05", "liquid_shortable").items()})
print("as of 2025-06-01:", store.get("2025-06-01", "liquid_shortable"))

Turnover grows with the noise at the filter edges: names near a threshold flip in and out. Hysteresis (enter above one level, leave below a lower one) or ranking on smoothed values reduces it. The store answers "what was the universe on that date, and from which screen config?" point-in-time, like the feature store of Part 10.

## Wrap-up

* Strategies are validated data: registered building blocks, trees, parameter ranges, a config hash on every trade.
* Screens compose small filters; measure the universe turnover they cause.
* Store screen results point-in-time, with the config that produced them.
* Graded version: `labs/part12/week41_integration` (`build_condition`, `validate_config`, `load_strategy`, the filters, `screen`, `universe_turnover`, `UniverseStore`).